# ۲ · تشخیص و جستجوی حیوانات — معماری ترکیبی

**MegaDetector** جای حیوان را پیدا می‌کند → **MobileNetV3-Large** آن را
شناسایی می‌کند → **CLIP** فقط وقتی می‌آید که گونه در واژگان ImageNet نباشد.

---

## چرا ترکیبی؟ عددها

| مدل | پارامتر | GFLOPs هر برش | دقت ImageNet | واژگان |
|:--|--:|--:|--:|:--|
| MobileNetV3-**Small** (قبلی) | ۲.۵M | ۰.۰۵۷ | ۶۷.۷٪ | ۳۹۸ گونه |
| **MobileNetV3-Large** (الان) | ۵.۵M | **۰.۲۱۷** | **۷۴.۰٪** | ۳۹۸ گونه |
| CLIP ViT-L/14 | ۴۲۸M | **~۸۱** | ~۷۵.۵٪ | **باز** |

CLIP برای هر برش **۳۷۳ برابر** محاسبه می‌خواهد، در ازای ~۱.۵٪ دقت بیشتر.
معامله‌ی بدی است — **وقتی که گونه در واژگان طبقه‌بند باشد**.

ولی وقتی نباشد، CLIP تنها گزینه است: `quokka` کلاس ImageNet نیست، پس
**هیچ** مدل supervised ای نمی‌تواند برش بگرداند.

**پس:** طبقه‌بند مسیر اصلی، CLIP پشتیبان.

---

## 🐞 باگ اصلی چه بود؟ (مهم: از MobileNetV3 نبود)

مدل **درست** جواب می‌داد:

```
عکس فیل → MobileNetV3 → index 386 → "African elephant"   ✅ درست
```

خرابی از لایه‌ی بعدی بود که ۱۰۰۰ اسم را به ۸ دسته می‌ریخت:

```python
if any(w in name_lower for w in ['spider','tick','ant','bee', ...]):
    return "Insect / Arthropod"     # ← شاخه ۵

elif any(w in name_lower for w in ['bear','elephant', ...]):
    return "Large Mammal"           # ← شاخه ۷ — هرگز به فیل نمی‌رسد
```

`'ant' in 'african elephant'` → **True**، چون داخل «eleph**ant**» است.
پایتون دنبال **زیررشته** می‌گشت، نه **کلمه**.

**راه‌حل واقعی:** آن لایه‌ی ۸ دسته‌ای را حذف کن و مستقیم با **نام واقعی
کلاس** تطبیق بده — با مرز کلمه (`\bant\b`). خودِ طبقه‌بند سالم بود.

## 🔬 اثبات باگ روی داده‌ی واقعی

این سلول منطق قدیمی را بازسازی می‌کند و روی لیست واقعی ImageNet اجرا.

In [ ]:
import re

import requests

classes = requests.get(
    "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
).text.strip().split("\n")

OLD_BUCKETS = [
    ("Insect / Arthropod", ["spider","tick","ant","bee","fly","beetle","moth"]),
    ("Large Mammal", ["bear","elephant","camel","ox","cow","bull","zebra","pig","hog"]),
    ("Cat", ["cat","lion","tiger","leopard","lynx"]),
    ("Reptile", ["snake","boa","python","lizard","turtle","frog"]),
    ("Small Mammal", ["mouse","rat","bat","rabbit","squirrel"]),
    ("Bird", ["bird","owl","cock","hen","eagle","duck"]),
    ("Aquatic", ["fish","eel","gar","crab","whale","seal"]),
]

def old_way(name):
    """روش قدیمی: زیررشته، بدون مرز کلمه."""
    low = name.lower()
    for cat, words in OLD_BUCKETS:
        for w in words:
            if w in low:                       # ← ریشه‌ی باگ
                return cat, w
    return "Object / Non-Animal", ""

def new_way(term, name):
    """روش جدید: مرز کلمه."""
    return any(re.search(rf"\b{re.escape(term)}\b", p.strip(), re.I)
               for p in name.split(","))

print("="*70)
print(f"{'کلاس ImageNet':<22} {'روش قدیمی':<24} {'مرز کلمه؟'}")
print("="*70)
for q, w in [("African elephant","ant"), ("Indian elephant","ant"),
             ("giant panda","ant"), ("wild boar","boa"),
             ("mailbox","ox"), ("bathtub","bat"), ("beer bottle","bee")]:
    if q in classes:
        cat, why = old_way(q)
        ok = new_way(w, q)
        print(f"{q:<22} ❌ {cat:<21} {'❌ match' if ok else '✅ no match'}")

print()
print("چرا؟  'ant' in 'african elephant'  →", "ant" in "african elephant")
print("      re.search(r'\\bant\\b', 'African elephant') →",
      bool(re.search(r"\bant\b", "African elephant")))
print()
print("یعنی: طبقه‌بند درست می‌گفت. آن لایه خرابش می‌کرد.")

## 🗺️ معماری

```
┌─ ایندکس (یک بار به‌ازای هر عکس) ──────────────────────────────┐
│                                                               │
│  عکس ──> MegaDetector ──> جعبه‌های حیوان                     │
│                 ↓                                             │
│           برش + پدینگ ۱۵٪ + کلمپ مرزی                        │
│                 ↓                                             │
│    MobileNetV3-Large ──> ۵ حدس برتر + احتمال                 │
│                 ↓                                             │
│    SQL:  animal_predictions(فایل, جعبه, class_id, احتمال)    │
└───────────────────────────────────────────────────────────────┘

┌─ پرس‌وجو ─────────────────────────────────────────────────────┐
│                                                               │
│  "zebra" ──> آیا در واژگان ImageNet هست؟                     │
│                                                               │
│     ✅ بله (index 340)              ❌ نه ("quokka")          │
│           ↓                              ↓                    │
│    ⚡ فقط SELECT                    CLIP روی برش‌ها          │
│    صفر استنتاج مدل                  (فقط این حالت)           │
└───────────────────────────────────────────────────────────────┘
```

**برد اصلی:** طبقه‌بندی به زمان ایندکس منتقل شد. پرس‌وجوی `"zebra"` حالا
یک `SELECT` است — هیچ مدلی اجرا نمی‌شود. در نسخه‌ی قبلی، هر پرس‌وجو
**MegaDetector و MobileNetV3 را از نو** روی همه‌ی کاندیداها اجرا می‌کرد.

In [ ]:
# ── نصب وابستگی‌ها ───────────────────────────────────────────────────
# ترتیب مهم است: insightface نسخه CPU از onnxruntime نصب می‌کند،
# پس اول آن را حذف و بعد نسخه GPU را نصب می‌کنیم.
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q insightface ultralytics transformers opencv-python tqdm
!pip uninstall -y onnxruntime -q
!pip install -q onnxruntime-gpu

import onnxruntime as ort
print("✅ نصب کامل شد")
print("موتورهای در دسترس:", ort.get_available_providers())

In [ ]:
# ── تشخیص دستگاه و دقت عددی ─────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    کد قبلی سه جور مقایسه داشت:
#        if device.type == "cuda":   ← درست
#        if device == "cuda":        ← همیشه False !
#
#    چون device یک شیء torch.device است، نه رشته.
#    تست شده روی torch 2.8:  torch.device('cuda') == 'cuda'  →  False
#
#    نتیجه: روی GPU مدل half می‌شد ولی ورودی float32 می‌ماند →
#    RuntimeError: expected scalar type Half but found Float
#
# ✅ راه‌حل: دستگاه و dtype را یکجا حل می‌کنیم تا نتوانند با هم اختلاف پیدا کنند.

import torch
from dataclasses import dataclass


@dataclass(frozen=True)
class Runtime:
    device: torch.device
    use_half: bool

    @property
    def is_cuda(self) -> bool:
        return self.device.type == "cuda"

    def cast_inputs(self, inputs: dict) -> dict:
        """انتقال ورودی به دستگاه با dtype هماهنگ.
        فقط تنسورهای اعشاری half می‌شوند؛ input_ids باید عدد صحیح بماند."""
        out = {}
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                v = v.to(self.device)
                if self.use_half and v.is_floating_point():
                    v = v.half()
            out[k] = v
        return out

    def prepare_model(self, model):
        model = model.to(self.device)
        if self.use_half:
            model = model.half()
        return model.eval()


def resolve_runtime(preference: str = "auto", half: bool = True) -> Runtime:
    name = preference
    if preference == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(name)
    # fp16 روی CPU کندتر از fp32 است و بعضی عملیات پشتیبانی نمی‌شوند
    return Runtime(device=device, use_half=half and device.type == "cuda")


RT = resolve_runtime()
print(f"🚀 دستگاه: {RT.device}   |   نیمه‌دقت (FP16): {RT.use_half}")

In [ ]:
# ── ابزار جعبه‌ها ───────────────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    برش بدون کلمپ مرزی:  image[y1-pad : y2+pad, x1-pad : x2+pad]
#    اگر x1-pad منفی شود، numpy خطا نمی‌دهد — بی‌صدا از ته آرایه
#    برمی‌دارد و برش غلط می‌دهد. یعنی حیوانی که به لبه کادر چسبیده،
#    از روی پیکسل‌های اشتباه طبقه‌بندی می‌شد.

def area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def overlap_fraction(box, other) -> float:
    """چه کسری از box داخل other است.

    عمداً نامتقارن: سؤال «چقدر از این کادرِ غذا روی صورت است»،
    نه «این دو کادر چقدر شبیه‌اند». IoU جواب سؤال اشتباه را می‌دهد —
    کادر کوچک غذا کاملاً داخل کادر بزرگ صورت، IoU پایینی دارد
    ولی overlap_fraction آن ۱.۰ است.
    """
    ax1, ay1, ax2, ay2 = box
    bx1, by1, bx2, by2 = other
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix1 >= ix2 or iy1 >= iy2:
        return 0.0
    a = area(box)
    return ((ix2 - ix1) * (iy2 - iy1) / a) if a else 0.0


def overlaps_any(box, others, threshold: float) -> bool:
    return any(overlap_fraction(box, o) > threshold for o in others)


def pad_box(box, width, height, ratio=0.0, pixels=0):
    """بزرگ‌کردن کادر + کلمپ به مرز تصویر (✅ رفع باگ اسلایس منفی)."""
    x1, y1, x2, y2 = box
    px = int(round((x2 - x1) * ratio)) + pixels
    py = int(round((y2 - y1) * ratio)) + pixels
    return (max(0, x1 - px), max(0, y1 - py),
            min(width, x2 + px), min(height, y2 + py))


def is_large_enough(box, min_px: int) -> bool:
    """رد کردن کادرهای ریز.

    🐞 قبلاً هیچ حداقلی نبود: یک کادر ۱۲×۹ پیکسل ۲۰ برابر بزرگ می‌شد
    به ۲۲۴×۲۲۴ و یک برچسب با درصد بالا می‌گرفت.
    """
    x1, y1, x2, y2 = box
    return (x2 - x1) >= min_px and (y2 - y1) >= min_px


print("✅ ابزار جعبه‌ها آماده (با کلمپ مرزی و حداقل اندازه)")

In [ ]:
# ── امتیازدهی رقابتی ────────────────────────────────────────────────
#
# 💡 بهترین ایده‌ی کل پروژه — و حالا در هر سه مسیر تشخیص استفاده می‌شود.
#
# به‌جای آستانه گذاشتن روی شباهت خام، پرسش را رقابتی می‌کنیم:
#
#     prompts = ["a photo of a zebra",                    ← فرضیه
#                "a photo of a different kind of animal", ← رقیب
#                "a photo of scenery with no animal"]     ← رقیب
#     score = softmax(logits)[0]
#
# چرا بهتر است؟ شباهت کسینوسی «فرضیه صفر» ندارد — هر برشی یک عددی
# می‌دهد و جای برش اصولی ندارد. softmax روی رقبا به مدل اجازه می‌دهد
# بگوید «هیچ‌کدام»، پس خروجی یک احتمال کالیبره است نه یک فاصله.
#
# شاهدش در همین پروژه: مسیر غذا (با رقیب) ۹۶–۹۹٪ می‌داد،
# مسیر متنی (کسینوس خام) برای بهترین نتیجه‌اش ۲۳٪ نشان می‌داد.

import math
from dataclasses import dataclass


def softmax(values):
    if not values:
        return []
    top = max(values)
    exps = [math.exp(v - top) for v in values]   # پایدار عددی
    total = sum(exps)
    return [e / total for e in exps]


def contrastive_score(logits) -> float:
    """احتمال پسین فرضیه در برابر رقبایش."""
    if not logits:
        raise ValueError("logits خالی")
    if len(logits) == 1:
        raise ValueError("حداقل یک prompt رقیب لازم است؛ "
                         "با یک کاندیدا softmax همیشه ۱.۰ است")
    return softmax(logits)[0]


@dataclass(frozen=True)
class Match:
    file_name: str
    file_path: str
    score: float
    box: tuple = None
    label: str = None


def best_per_image(matches):
    """یک نتیجه به‌ازای هر تصویر — قوی‌ترین نمونه.

    🐞 باگی که رفع شد:
       نسخه یکپارچه break هر تصویر را از دست داده بود، پس عکسی با
       دو چهره‌ی منطبق دو بار در نتایج ظاهر می‌شد.

       ضمناً کد قبلی «اولین» نمونه را ثبت می‌کرد و همان را کلید
       مرتب‌سازی می‌کرد — پس عکسی که گربه دومش ۹۵٪ بود با ۲۶٪
       رتبه‌بندی می‌شد.
    """
    best = {}
    for m in matches:
        cur = best.get(m.file_name)
        if cur is None or m.score > cur.score:
            best[m.file_name] = m
    return sorted(best.values(), key=lambda m: m.score, reverse=True)


assert abs(sum(softmax([1.0, 2.0, 3.0])) - 1.0) < 1e-9
assert contrastive_score([10.0, 0.0, 0.0]) > 0.95
assert contrastive_score([0.0, 10.0, 0.0]) < 0.05
print("✅ امتیازدهی رقابتی آماده")

## ⚙️ تنظیمات

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class AnimalConfig:
    gallery: str = "/content/gallery"
    database: str = "animal_index.db"

    animal_class_id: int = 0        # تنسور خام YOLOv5: ۰=حیوان ۱=انسان ۲=خودرو

    # 🐞 قبلاً وارونه بود: گیت ۰.۴۰ و جستجو ۰.۳۰.
    #    گیت یک فیلتر سخت است؛ تصویری که رد کند، جستجو هرگز نمی‌بیندش.
    gate_confidence: float = 0.30
    search_confidence: float = 0.40

    padding_ratio: float = 0.15
    min_crop_px: int = 40

    # ── مسیر اصلی: طبقه‌بند ──
    classifier_top_k: int = 5       # چند حدس برتر ذخیره شود
    classifier_threshold: float = 0.25

    # ── پشتیبان: CLIP ──
    clip_threshold: float = 0.55
    negative_prompts: tuple = (
        "a photo of a different kind of animal",
        "a photo of scenery with no animal in it",
    )

    def clip_prompts(self, q):
        return [f"a photo of a {q}", *self.negative_prompts]

CFG = AnimalConfig()
assert CFG.gate_confidence <= CFG.search_confidence
print("✅ تنظیمات معتبر — گیت", CFG.gate_confidence, "≤ جستجو", CFG.search_confidence)

## 🔤 حل پرس‌وجو — قلب معماری ترکیبی

این تابع تصمیم می‌گیرد کدام موتور جواب بدهد. اگر بتواند پرس‌وجو را به
کلاس‌های ImageNet نگاشت کند، مسیر ارزان انتخاب می‌شود.

**ترتیب اولویت مهم است:**

1. **نام دقیق** — `"lion"` باید فقط `lion` بدهد، نه `sea lion`
2. **عبارت عمومی** — `"dog"` باید همه‌ی نژادها را بدهد
3. **تطبیق جزئی با مرز کلمه** — `"terrier"` → همه‌ی تریرها
4. **حل‌نشده** → CLIP

In [ ]:
GENERIC_TERMS = {
    "dog": ("terrier","retriever","hound","spaniel","collie","poodle","pug",
            "bulldog","husky","malamute","chihuahua","beagle","dalmatian",
            "rottweiler","schnauzer","setter","pointer","sheepdog","mastiff",
            "pinscher","papillon","Pekinese","chow","keeshond","Samoyed",
            "Pomeranian","basenji","boxer","shepherd"),
    "cat": ("tabby","Persian cat","Siamese cat","Egyptian cat"),
    "big cat": ("lion","tiger","leopard","snow leopard","jaguar","cheetah",
                "cougar","lynx"),
    "elephant": ("African elephant","Indian elephant","tusker"),
    "bear": ("brown bear","American black bear","ice bear","sloth bear"),
    "monkey": ("macaque","langur","baboon","guenon","colobus","marmoset",
               "capuchin","howler monkey","titi","spider monkey","squirrel monkey"),
    "ape": ("gorilla","chimpanzee","orangutan","gibbon","siamang"),
}

IRREGULAR = {"wolves":"wolf","geese":"goose","mice":"mouse","oxen":"ox",
             "sheep":"sheep","deer":"deer","fish":"fish","foxes":"fox",
             "octopi":"octopus","butterflies":"butterfly"}

def normalise(q):
    q = " ".join(q.lower().split())
    if q in IRREGULAR:
        return IRREGULAR[q]
    if q.endswith("ies") and len(q) > 4:
        return q[:-3] + "y"
    if q.endswith("es") and len(q) > 4 and q[-3] in "sxzh":
        return q[:-2]
    if q.endswith("s") and not q.endswith("ss") and len(q) > 3:
        return q[:-1]
    return q

def word_match(term, class_name):
    """مرز کلمه — همان چیزی که باگ فیل را رفع می‌کند."""
    return any(re.search(rf"\b{re.escape(term)}\b", p.strip(), re.I)
               for p in class_name.split(","))

def exact_name(term, class_name):
    """آیا term دقیقاً یکی از نام‌های این کلاس است؟"""
    return any(term == p.strip().lower() for p in class_name.split(","))

def resolve(query):
    """پرس‌وجو → اندیس‌های ImageNet. تهی یعنی: برو سراغ CLIP."""
    n = normalise(query)

    # ۱. نام دقیق  ("lion" → lion، نه sea lion)
    ids = [i for i, c in enumerate(classes) if exact_name(n, c)]
    if ids:
        return ids, "exact"

    # ۲. عبارت عمومی  ("dog" → همه نژادها)
    if n in GENERIC_TERMS:
        s = set()
        for t in GENERIC_TERMS[n]:
            s.update(i for i, c in enumerate(classes) if word_match(t, c))
        s.update(i for i, c in enumerate(classes) if word_match(n, c))
        if s:
            return sorted(s), "generic"

    # ۳. تطبیق جزئی  ("terrier" → همه تریرها)
    ids = [i for i, c in enumerate(classes) if word_match(n, c)]
    if ids:
        return ids, "partial"

    # ۴. خارج از واژگان → CLIP
    return [], "unresolved"


# ── تست سریع ──
for q in ["zebra", "lion", "sea lion", "elephant", "dog", "terrier", "quokka"]:
    ids, via = resolve(q)
    names = [classes[i] for i in ids[:4]]
    engine = "MobileNetV3 ⚡" if ids else "CLIP 🐢"
    print(f"{q:<12} {via:<11} {len(ids):>3} کلاس  {engine:<15} {names}")

## 🧠 بارگذاری مدل‌ها

In [ ]:
import os
import urllib.request

import torch
from torchvision import models, transforms

# ── ۱. یابنده: MegaDetector v5a (yolov5 pin شده) ──
MD = "md_v5a.0.0.pt"
if not os.path.exists(MD):
    urllib.request.urlretrieve(
        "https://github.com/ecologize/CameraTraps/releases/download/v5.0/md_v5a.0.0.pt", MD)
detector = torch.hub.load("ultralytics/yolov5:v7.0", "custom",
                          path=MD, trust_repo=True).to(RT.device)
print("✅ MegaDetector v5a")

# ── ۲. طبقه‌بند: MobileNetV3-Large (ارتقا از Small) ──
weights = models.MobileNet_V3_Large_Weights.DEFAULT
classifier = models.mobilenet_v3_large(weights=weights).to(RT.device).eval()
print(f"✅ MobileNetV3-Large — {weights.meta['_metrics']['ImageNet-1K']['acc@1']}% top-1, "
      f"{weights.meta['num_params']:,} پارامتر")

# 🐞 باگی که رفع شد: کد قبلی Resize((224,224)) استفاده می‌کرد که نسبت
#    تصویر را له می‌کند — بدترین حالت برای برش‌های غیرمربع مثل زرافه
#    (بلند) یا داکسهوند (پهن). زنجیره درست در نوت‌بوک اصلی نوشته شده
#    بود ولی هرگز استفاده نشد.
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),        # ✅ نسبت حفظ می‌شود
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ── ۳. پشتیبان: CLIP (فقط وقتی لازم شود بارگذاری می‌شود) ──
_clip = {}
def get_clip():
    """بارگذاری تنبل — ۱.۷ گیگابایت فقط وقتی واقعاً لازم است."""
    if not _clip:
        from transformers import CLIPModel, CLIPProcessor
        name = "openai/clip-vit-large-patch14"
        _clip["model"] = RT.prepare_model(CLIPModel.from_pretrained(name))
        _clip["proc"] = CLIPProcessor.from_pretrained(name)
        print("✅ CLIP بارگذاری شد (پشتیبان)")
    return _clip["model"], _clip["proc"]

## 📥 ایندکس‌گذاری — طبقه‌بندی اینجا انجام می‌شود

این مهم‌ترین تغییر معماری است: MobileNetV3 **یک بار** روی هر برش اجرا
می‌شود و نتیجه ذخیره می‌شود. پرس‌وجو بعداً فقط `SELECT` می‌زند.

In [ ]:
import sqlite3
from contextlib import contextmanager
from pathlib import Path

import cv2
from tqdm.auto import tqdm

SCHEMA = """
CREATE TABLE IF NOT EXISTS gallery_meta (
    file_name  TEXT PRIMARY KEY,
    file_path  TEXT NOT NULL,
    has_animal INTEGER NOT NULL DEFAULT 0
);
CREATE TABLE IF NOT EXISTS animal_predictions (
    file_name   TEXT NOT NULL,
    bbox        TEXT NOT NULL,
    rank        INTEGER NOT NULL,
    class_id    INTEGER NOT NULL,
    probability REAL NOT NULL,
    PRIMARY KEY (file_name, bbox, rank)
);
CREATE INDEX IF NOT EXISTS idx_has_animal ON gallery_meta(has_animal);
CREATE INDEX IF NOT EXISTS idx_class ON animal_predictions(class_id);
"""

@contextmanager
def connect():
    conn = sqlite3.connect(CFG.database)
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()

with connect() as c:
    c.executescript(SCHEMA)


def detect_animals(img_rgb, conf):
    with torch.no_grad():
        preds = detector(img_rgb).xyxy[0].cpu().numpy()
    return [((int(r[0]),int(r[1]),int(r[2]),int(r[3])), float(r[4]))
            for r in preds
            if int(r[5]) == CFG.animal_class_id and float(r[4]) >= conf]


def classify_crops(crops, top_k):
    """MobileNetV3 روی دسته‌ای از برش‌ها — یک forward، نه یکی‌یکی."""
    if not crops:
        return []
    batch = torch.stack([preprocess(c) for c in crops]).to(RT.device)
    with torch.no_grad():
        probs = torch.softmax(classifier(batch), dim=1)
    top = torch.topk(probs, k=min(top_k, probs.shape[1]), dim=1)
    return [list(zip(idx.tolist(), val.tolist()))
            for idx, val in zip(top.indices, top.values)]


def build_index():
    paths = sorted(p for p in Path(CFG.gallery).iterdir()
                   if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"})
    n_img = n_box = 0

    with connect() as conn:
        for path in tqdm(paths, desc="Indexing"):
            img = cv2.imread(str(path))
            if img is None:
                continue
            h, w = img.shape[:2]
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            boxes = [(b, c) for b, c in detect_animals(rgb, CFG.gate_confidence)
                     if is_large_enough(b, CFG.min_crop_px)]

            crops, kept = [], []
            for box, conf in boxes:
                pb = pad_box(box, w, h, ratio=CFG.padding_ratio)
                patch = rgb[pb[1]:pb[3], pb[0]:pb[2]]
                if patch.size:
                    crops.append(patch)
                    kept.append((box, conf))

            # ⚡ طبقه‌بندی همین‌جا، نه در زمان پرس‌وجو
            preds = classify_crops(crops, CFG.classifier_top_k)

            conn.execute("DELETE FROM animal_predictions WHERE file_name=?", (path.name,))
            conn.execute("INSERT OR REPLACE INTO gallery_meta VALUES (?,?,?)",
                         (path.name, str(path), int(bool(kept))))
            for (box, conf), top in zip(kept, preds):
                key = ",".join(map(str, box))
                for rank, (cid, prob) in enumerate(top):
                    conn.execute("INSERT OR REPLACE INTO animal_predictions VALUES (?,?,?,?,?)",
                                 (path.name, key, rank, cid, prob))
                n_box += 1
            n_img += 1

    print(f"✅ {n_img} تصویر، {n_box} حیوان طبقه‌بندی و ذخیره شد")

# build_index()

## 🔎 جستجو — مسیریابی خودکار

`search_animal` خودش تصمیم می‌گیرد کدام موتور را صدا بزند.

In [ ]:
import time

from PIL import Image


def _search_indexed(ids, query):
    """⚡ مسیر ارزان: فقط SQL، هیچ مدلی اجرا نمی‌شود."""
    placeholders = ",".join("?" * len(ids))
    with connect() as conn:
        rows = conn.execute(
            f"""SELECT p.file_name, g.file_path, p.bbox, p.class_id, p.probability
                FROM animal_predictions p JOIN gallery_meta g USING(file_name)
                WHERE p.class_id IN ({placeholders}) AND p.probability >= ?""",
            (*ids, CFG.classifier_threshold)).fetchall()

    return best_per_image([
        Match(name, path, prob, box=tuple(int(v) for v in bbox.split(",")),
              label=classes[cid])
        for name, path, bbox, cid, prob in rows])


def _search_clip(query):
    """🐢 مسیر گران: فقط برای گونه‌های خارج از واژگان."""
    model, proc = get_clip()
    with connect() as conn:
        rows = conn.execute("SELECT file_name, file_path FROM gallery_meta "
                            "WHERE has_animal=1").fetchall()
        boxes = {}
        for name, bbox in conn.execute(
                "SELECT DISTINCT file_name, bbox FROM animal_predictions"):
            boxes.setdefault(name, []).append(tuple(int(v) for v in bbox.split(",")))

    cands, crops = [], []
    for name, path in rows:
        img = cv2.imread(path)
        if img is None:
            continue
        h, w = img.shape[:2]
        for box in boxes.get(name, []):
            pb = pad_box(box, w, h, ratio=CFG.padding_ratio)
            patch = img[pb[1]:pb[3], pb[0]:pb[2]]
            if patch.size:
                cands.append((name, path, box))
                crops.append(Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)))

    if not crops:
        return []
    inputs = proc(text=CFG.clip_prompts(query), images=crops,
                  return_tensors="pt", padding=True)
    inputs = RT.cast_inputs(dict(inputs))
    with torch.no_grad():
        logits = model(**inputs).logits_per_image.float().cpu().tolist()

    return best_per_image([
        Match(n, p, contrastive_score(r), box=b, label=query)
        for (n, p, b), r in zip(cands, logits)
        if contrastive_score(r) >= CFG.clip_threshold])


def search_animal(query):
    t0 = time.time()
    ids, via = resolve(query)

    if ids:
        print(f"🔀 '{query}' → {len(ids)} کلاس ImageNet ({via}) → "
              f"⚡ MobileNetV3 از ایندکس، صفر استنتاج")
        results = _search_indexed(ids, query)
    else:
        print(f"🔀 '{query}' خارج از واژگان ImageNet → 🐢 CLIP روی برش‌ها")
        results = _search_clip(query)

    print(f"⏱  {time.time()-t0:.4f} ثانیه — {len(results)} نتیجه")
    for i, m in enumerate(results, 1):
        print(f" [{i:>2}] {m.file_name:<26} {m.score*100:6.2f}%  {m.label}")
    return results


# search_animal("zebra")        # ⚡ مسیر ارزان
# search_animal("dog")          # ⚡ همه نژادها
# search_animal("quokka")       # 🐢 CLIP

## 📋 خلاصه

### معماری

| مرحله | مدل | کِی اجرا می‌شود |
|:--|:--|:--|
| یافتن حیوان | MegaDetector v5a | زمان ایندکس |
| شناسایی گونه | **MobileNetV3-Large** | **زمان ایندکس** (ذخیره می‌شود) |
| گونه خارج از واژگان | CLIP ViT-L/14 | فقط زمان پرس‌وجو، فقط اگر لازم شود |

### قبل → بعد

| | قبلاً | الان |
|:--|:--|:--|
| طبقه‌بند | MobileNetV3-**Small** (۶۷.۷٪) | MobileNetV3-**Large** (۷۴.۰٪) |
| نگاشت گونه | ۸ دسته با زیررشته — **خراب** | تطبیق با نام واقعی + مرز کلمه |
| فیل | «حشره» ❌ | `African elephant` ✅ |
| زمان طبقه‌بندی | هر پرس‌وجو | **یک بار، در ایندکس** |
| MegaDetector | هر پرس‌وجو دوباره | یک بار، جعبه‌ها ذخیره |
| گونه خارج از واژگان | غیرممکن | CLIP |
| آستانه اطمینان | ندارد | ۰.۲۵ (طبقه‌بند) / ۰.۵۵ (CLIP) |
| گیت/جستجو | ۰.۴۰ / ۰.۳۰ وارونه | ۰.۳۰ / ۰.۴۰ + assert |
| پیش‌پردازش | `Resize((224,224))` له‌کننده | `Resize(256)+CenterCrop(224)` |
| yolov5 | master بدون pin | `v7.0` |

### چرا این بهتر از هر دو حالت قبلی است

- **از نسخه‌ی اصلی بهتر:** نگاشت خراب حذف شد، طبقه‌بند قوی‌تر شد،
  طبقه‌بندی به ایندکس منتقل شد
- **از پیشنهاد اول من بهتر:** CLIP فقط وقتی اجرا می‌شود که واقعاً لازم
  باشد، نه برای هر برش با ۳۷۳ برابر هزینه